In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

In [2]:
train_dir = r'D:\new_prostate_vis\train'
test_dir =  r'D:\new_prostate_vis\test'
valid_dir = r'D:\new_prostate_vis\Valid'

In [3]:
import tensorflow.keras.backend as K 

from tensorflow.keras.models import Model

from tensorflow.keras.layers import *
import collections
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import regularizers

In [4]:
batch_size = 4
img_height = 256
img_width = 256
no_of_classes = 4
classes_name = [0,1,2,3]
input_shape = (img_height , img_width , 3)
datagen = ImageDataGenerator(
 rescale=1. / 255,
 featurewise_center=True,
 horizontal_flip = False,
 vertical_flip = False,
 #validation_split = 0.1,
 featurewise_std_normalization=True)
train_generator = datagen.flow_from_directory(
 train_dir,
 target_size=(img_height, img_width),
 batch_size=batch_size,
 shuffle = True,
 class_mode='categorical')
validation_generator = datagen.flow_from_directory(
 valid_dir,
 target_size=(img_height, img_width),
 batch_size=batch_size,
 shuffle = True,
 class_mode='categorical')
# print(train_generator[0])
print("Trainging classes")
print(train_generator.class_indices)
print("Trainging Labels")
print(train_generator.labels)
print("Validation classes")
print(validation_generator.class_indices)
print("Validation Labels")
print(validation_generator.labels)

Found 16000 images belonging to 4 classes.
Found 4000 images belonging to 4 classes.
Trainging classes
{'0': 0, '1': 1, '2': 2, '3': 3}
Trainging Labels
[0 0 0 ... 3 3 3]
Validation classes
{'0': 0, '1': 1, '2': 2, '3': 3}
Validation Labels
[0 0 0 ... 3 3 3]


In [5]:
def res_block(x,f): 
    w,h,c = x.shape[1],x.shape[2],x.shape[3]
    x41 = AveragePooling2D(pool_size = (w,h))(x)
    x41 = Conv2D(f, (1, 1), activation='sigmoid', padding='same')(x41)
    x4 = Multiply()([x,x41])
    return x4

In [6]:
def conv_layer(x,f):
    conv1 = tf.keras.layers.Conv2D(f, kernel_size=(3, 3), padding='same')(x)
    norm1 = tf.keras.layers.BatchNormalization()(conv1)
    relu1 = LeakyReLU()(norm1)
    return relu1

def block(x,y):
    _, width, height, channels = x.get_shape().as_list()
    group_ch = channels // 2
    x = Reshape([width, height, group_ch, 2])(x)
    x = Permute([1, 2, 4, 3])(x)
    x0 = x[:,:,:,0]
    x1 = x[:,:,:,1]
    
    _, width, height, channels = y.get_shape().as_list()
    group_ch = channels // 2
    y = Reshape([width, height, group_ch, 2])(y)
    y = Permute([1, 2, 4, 3])(y)
    y0 = y[:,:,:,0]
    y1 = y[:,:,:,1]
    
    xy0 = Concatenate()([x0,y1])
    xy1 = Concatenate()([y0,x1])
        
    xy2 = conv_layer(xy0, channels)
    xy3 = conv_layer(xy1, channels) 
    x_o = res_block(xy2, channels)
    y_o = res_block(xy3, channels)
    x_out = Add()([x_o,y_o]) 
    return x_out

In [7]:
def resnet_block(block_input, num_filters):
    conv1 = conv_layer(block_input,num_filters)
    conv2 = block(block_input,conv1)
    sum = Add()([conv2,block_input])
    relu2 = tf.keras.layers.Activation('relu')(sum)
    return relu2


In [8]:


def efficient_block(ip, filters):

    shortcut = ip
    ip = tf.keras.layers.GlobalAveragePooling2D()(ip)
    
    a=Reshape((1,1,filters))(ip)
    
    x = Conv2D(filters//4, 1, padding='same')(a)
    x = BatchNormalization()(x)
    x = LeakyReLU()(x)
    x = Conv2D(filters, 1, padding='same',activation='sigmoid')(x)

    y = Conv2D(filters//2, 1, padding='same')(a)
    y = BatchNormalization()(y)
    y = LeakyReLU()(y)
    y = Conv2D(filters, 1, padding='same',activation='sigmoid')(y)

    ad=Add()([x,y])
    gp= tf.keras.layers.GlobalAveragePooling2D()(ad)
    
    alphaeff = alpha_effi(ad,filters,gp)
    
    senw = senew(ad,filters,gp)
    ad=Multiply()([alphaeff,senw])
    r = multiply([shortcut,ad ])
    
    return r


In [9]:
def alpha_effi(ip,filters,gp):
    shortcut = ip
    # ip = layers.GlobalAveragePooling2D()(ip)
    
    a=Reshape((1,1,filters))(gp)
    
    x = Conv2D(filters, 1, padding='same')(a)
    y = Conv2D(filters//4, 1, padding='same')(a)
    z = Conv2D(filters//8, 1, padding='same')(a)
    
    cnct = Concatenate()([x,y,z])
    # bn = layers.BatchNormalization()(cnct)
    r1 = LeakyReLU()(cnct)
    d1 = Dense(filters,activation='sigmoid')(r1)
    
    res = multiply([shortcut,d1 ])
   
    return res

    

In [10]:
def senew(ip,f,gp):
    r=4
    shortcut = ip
    # ip = layers.GlobalAveragePooling2D()(ip)
    
    a=Reshape((1,1,f))(gp)
    
    x = Dense(f/r, activation='relu')(a)
    x = Dense(f, activation='sigmoid')(x)
    res = Add()([x,shortcut])
   
    return res

In [11]:

def ProsCan43():

  input = tf.keras.layers.Input(shape=(256, 256, 3))
  eip1 = Conv2D(16,1)(input)  
  ef1 = efficient_block(eip1,16)
    
  conv1 = conv_layer(input,16)
  block1 = resnet_block(conv1, 16)
  block2 = resnet_block(block1, 16)

  ad1=Add()([ef1,block2])
  pool1 = tf.keras.layers.MaxPooling2D((2, 2), (2,2))(ad1)

  eip2 = Conv2D(32,1)(input) 
  ef2 = efficient_block(eip2,32)
  
  
  plef2 = tf.keras.layers.MaxPooling2D((2, 2), (2,2))(ef2)
  conv2 = conv_layer(pool1,32)
  block4 = resnet_block(conv2, 32)
  block5 = resnet_block(block4, 32)
  
  ad2=Add()([plef2,block5])
  pool2 = tf.keras.layers.MaxPooling2D((2, 2), (2,2))(ad2)

  conv3 = conv_layer(pool2,64)
  block7 = resnet_block(conv3, 64)
  block8 = resnet_block(block7, 64)

  conip=Conv2D(64,1)(input)
  plip = tf.keras.layers.MaxPooling2D((2, 2), (4,4))(conip)
  
  ad3=Add()([plip,block8])
  pool3 = tf.keras.layers.MaxPooling2D((1, 1), (1,1))(ad3)
  
  global_pool = tf.keras.layers.GlobalAveragePooling2D()(pool3)
  
  x = Dense(128, activation='relu')(global_pool)
  x = LeakyReLU()(x)
  x = Dense(64, activation='relu')(x)
  x = LeakyReLU()(x)
  output = tf.keras.layers.Dense(4, activation='softmax')(x)
  model = tf.keras.models.Model(inputs=input, outputs=output)
  return model

In [12]:
model = ProsCan43()
model.compile(optimizer = 'Adam' , loss = 'categorical_crossentropy' , metrics = ["accuracy"])
model.summary()


Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 256, 256, 3) 0                                            
__________________________________________________________________________________________________
conv2d_8 (Conv2D)               (None, 256, 256, 16) 448         input_1[0][0]                    
__________________________________________________________________________________________________
batch_normalization_2 (BatchNor (None, 256, 256, 16) 64          conv2d_8[0][0]                   
__________________________________________________________________________________________________
leaky_re_lu_3 (LeakyReLU)       (None, 256, 256, 16) 0           batch_normalization_2[0][0]      
______________________________________________________________________________________________

In [13]:
import time

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor = 'val_accuracy' , mode='max' ,
                                                  factor = 0.5 , patience = 10 , verbose=1 , cooldown = 1,
                                                 min_delta = 0.0001)

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', min_delta=0.0001, patience=30, verbose=1,
                                              mode = 'max', restore_best_weights = True)
check_path = 'F:/vishnu/weights/new3.h5'
checkpoint = tf.keras.callbacks.ModelCheckpoint(check_path, monitor = 'val_accuracy', verbose=1, save_best_only=True, save_weights_only=True, mode='max')

t10 = time.time()
history_1 = model.fit(x=train_generator , validation_data = validation_generator ,
                                  steps_per_epoch= len(train_generator) ,
                                  validation_batch_size = len(validation_generator)
                                  ,epochs = 100,callbacks = [reduce_lr, early_stop, checkpoint] )
t11 = time.time()
time_forloop = t11 - t10
print(time_forloop)

C:\Users\NITK\anaconda3\envs\Shyam_Lal\lib\site-packages\keras_preprocessing\image\image_data_generator.py:720: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn('This ImageDataGenerator specifies '
C:\Users\NITK\anaconda3\envs\Shyam_Lal\lib\site-packages\keras_preprocessing\image\image_data_generator.py:728: UserWarning: This ImageDataGenerator specifies `featurewise_std_normalization`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn('This ImageDataGenerator specifies '


Epoch 1/100
4000/4000 [==============================] - 454s 111ms/step - loss: 0.7356 - accuracy: 0.6634 - val_loss: 0.6046 - val_accuracy: 0.7147

Epoch 00001: val_accuracy improved from -inf to 0.71475, saving model to F:/vishnu/weights\new3.h5
Epoch 2/100
4000/4000 [==============================] - 448s 112ms/step - loss: 0.5639 - accuracy: 0.7290 - val_loss: 0.5781 - val_accuracy: 0.7030

Epoch 00002: val_accuracy did not improve from 0.71475
Epoch 3/100
4000/4000 [==============================] - 449s 112ms/step - loss: 0.5085 - accuracy: 0.7596 - val_loss: 0.9756 - val_accuracy: 0.6400

Epoch 00003: val_accuracy did not improve from 0.71475
Epoch 4/100
4000/4000 [==============================] - 449s 112ms/step - loss: 0.4657 - accuracy: 0.7802 - val_loss: 0.3966 - val_accuracy: 0.8190

Epoch 00004: val_accuracy improved from 0.71475 to 0.81900, saving model to F:/vishnu/weights\new3.h5
Epoch 5/100
4000/4000 [==============================] - 449s 112ms/step - loss: 0.4378 -

4000/4000 [==============================] - 442s 111ms/step - loss: 0.0612 - accuracy: 0.9771 - val_loss: 0.1849 - val_accuracy: 0.9625

Epoch 00075: val_accuracy did not improve from 0.96900
Epoch 76/100
4000/4000 [==============================] - 442s 111ms/step - loss: 0.0594 - accuracy: 0.9779 - val_loss: 0.2118 - val_accuracy: 0.9542

Epoch 00076: val_accuracy did not improve from 0.96900
Epoch 77/100
4000/4000 [==============================] - 444s 111ms/step - loss: 0.0612 - accuracy: 0.9779 - val_loss: 0.1633 - val_accuracy: 0.9660

Epoch 00077: val_accuracy did not improve from 0.96900
Epoch 78/100
4000/4000 [==============================] - 444s 111ms/step - loss: 0.0569 - accuracy: 0.9789 - val_loss: 0.2411 - val_accuracy: 0.9603

Epoch 00078: val_accuracy did not improve from 0.96900
Epoch 79/100
4000/4000 [==============================] - 446s 112ms/step - loss: 0.0561 - accuracy: 0.9808 - val_loss: 0.2206 - val_accuracy: 0.9645

Epoch 00079: val_accuracy did not impr

In [13]:
model.load_weights('F:/vishnu/weights/new3.h5')

In [14]:
test_d = ImageDataGenerator(rescale=1. / 255)
test = test_d.flow_from_directory(
    test_dir,
    target_size=(256,256),
    batch_size=1,
    shuffle = False,
    class_mode='categorical')

Found 5000 images belonging to 4 classes.


In [16]:
import numpy as np
test_step = test.n//test.batch_size
test.reset()
pred = model.predict_generator(test , steps = test_step , verbose = 1)
pred_class_indices = np.argmax(pred,axis=1)

## printing predicted labels
print(pred_class_indices)

5000/5000 [==============================] - 174s 35ms/step
[0 0 0 ... 3 3 3]


In [17]:
from sklearn.metrics import *
classes = [0,1,2,3]


for cl in classes:

    print("class: ",cl)

    a1 = np.uint8(test.labels == cl)
    a2 = np.uint8(pred_class_indices == cl)

    print('Accuracy {}'.format(accuracy_score(y_true=a1, y_pred=a2)))
    print('F1 {}'.format(f1_score(y_true=a1, y_pred=a2)))
    print('precision {}'.format(precision_score(y_true=a1, y_pred=a2)))
    print('recall {}'.format(recall_score(y_true=a1, y_pred=a2)))

    print('jaccard {}'.format(jaccard_score(y_true=a1, y_pred=a2)))
    print("_______________________________")

class:  0
Accuracy 0.9586
F1 0.9176938369781311
precision 0.9122529644268774
recall 0.9232
jaccard 0.8479059515062454
_______________________________
class:  1
Accuracy 0.9482
F1 0.8952689041649818
precision 0.9051512673753066
recall 0.8856
jaccard 0.8103953147877013
_______________________________
class:  2
Accuracy 0.9802
F1 0.9605420486249502
precision 0.9571088165210484
recall 0.964
jaccard 0.924079754601227
_______________________________
class:  3
Accuracy 0.9994
F1 0.9988014382740711
precision 0.9976057462090981
recall 1.0
jaccard 0.9976057462090981
_______________________________


In [18]:
print('Accuracy {}'.format(accuracy_score(y_true=test.labels, y_pred=pred_class_indices)))
print('F1 {}'.format(f1_score(y_true=test.labels, y_pred=pred_class_indices,average = "macro")))
print('precision {}'.format(precision_score(y_true=test.labels, y_pred=pred_class_indices,average = "macro")))
print('recall {}'.format(recall_score(y_true=test.labels, y_pred=pred_class_indices,average = "macro")))

print('jaccard {}'.format(jaccard_score(y_true=test.labels, y_pred=pred_class_indices,average = "macro")))
print('confusion_matrix\n {}'.format(confusion_matrix(y_true=test.labels, y_pred=pred_class_indices)))
print('classification_report\n {}'.format(classification_report(y_true=test.labels, y_pred=pred_class_indices)))
print('\n\n')

Accuracy 0.9432
F1 0.9430765570105335
precision 0.9430296986330826
recall 0.9432
jaccard 0.894996691776068
confusion_matrix
 [[1154   80   15    1]
 [ 104 1107   39    0]
 [   7   36 1205    2]
 [   0    0    0 1250]]
classification_report
               precision    recall  f1-score   support

           0       0.91      0.92      0.92      1250
           1       0.91      0.89      0.90      1250
           2       0.96      0.96      0.96      1250
           3       1.00      1.00      1.00      1250

    accuracy                           0.94      5000
   macro avg       0.94      0.94      0.94      5000
weighted avg       0.94      0.94      0.94      5000






In [19]:
import tensorflow as tf
from tensorflow.python.profiler.model_analyzer import profile
from tensorflow.python.profiler.option_builder import ProfileOptionBuilder

def get_flops(model):
  forward_pass = tf.function(model.call, input_signature=[tf.TensorSpec(shape=(1,) + model.input_shape[1:])])
  graph_info = profile(forward_pass.get_concrete_function().graph, options=ProfileOptionBuilder.float_operation())
  flops = graph_info.total_float_ops
  return flops

flops = get_flops(model)
print(f"FLOPs: {flops / 1e+9:,} G")

Instructions for updating:
Use `tf.compat.v1.graph_util.tensor_shape_from_node_def_name`
FLOPs: 5.884960218 G
